# Streaming i asynchroniczność w LangGraph

Notebook oparty na **2. LLMy w Langgraph**.

## Kluczowa zasada

> **Pipeline budujemy zawsze tak samo.** Różnica leży wyłącznie w sposobie wywołania.

| | Zwraca wynik na końcu | Zwraca wynik po każdym nodzie |
|---|---|---|
| **Synchroniczny** (blokuje wątek) | `invoke` | `stream` |
| **Asynchroniczny** (zwalnia wątek) | `ainvoke` | `astream` |

### Porównanie:
- **sync vs async** — czy wątek jest zajęty podczas czekania na I/O (np. odpowiedź LLM)
- **invoke vs stream** — czy dostajesz jeden wynik na końcu, czy chunk po każdym nodzie

### Gdzie ma znaczenie async?
W tym notebooku wywołujemy pipeline jeden raz — async nic tu nie przyspiesza.
Async ma znaczenie np. w przypadku opakowania grafu w aplikację webową. Gdy wiele requestów przychodzi jednocześnie,
wersja async zwalnia wątek podczas czekania na LLM i pozwala obsługiwać inne requesty.
Wersja sync blokuje cały serwer do czasu odpowiedzi.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## State

Taki sam jak w notebooku 2.

In [ ]:
from typing import TypedDict


class State(TypedDict):
    question: str
    prompt_messages: list[dict]
    llm_response: str
    formatted_output: str

## Node'y

Node'y są zwykłymi `def` — dzięki temu działają ze wszystkimi czterema metodami wywołania (`invoke`, `ainvoke`, `stream`, `astream`).

> `async def` node działa **tylko** z `ainvoke` i `astream`. Zwykłe `def` działa ze wszystkimi czterema — gdy wywołasz `ainvoke`/`astream`, LangGraph sam opakuje node w executor i wywoła go asynchronicznie.

Każdy node ma sztuczny `time.sleep()` symulujący pracę (np. wywołanie API). Dzięki temu wyraźnie widać, kiedy każdy node kończy działanie.

In [ ]:
import time
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI


def build_prompt_node(state: State, config: RunnableConfig):
    time.sleep(3)
    configurable = config.get("configurable", {})
    system_prompt = configurable.get(
        "system_prompt",
        "Jesteś pomocnym asystentem. Odpowiadaj zwięźle i rzeczowo po polsku."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": state["question"]},
    ]
    return {"prompt_messages": messages}


def call_llm_node(state: State, config: RunnableConfig):
    time.sleep(4)
    configurable = config.get("configurable", {})
    model_name = configurable.get("model_name", "openai/gpt-4o-mini")
    temperature = configurable.get("temperature", 0.7)

    llm = ChatOpenAI(model=model_name, temperature=temperature)
    response = llm.invoke(state["prompt_messages"])
    return {"llm_response": response.content}


def format_output_node(state: State, config: RunnableConfig):
    time.sleep(2)
    configurable = config.get("configurable", {})
    model_name = configurable.get("model_name", "openai/gpt-4o-mini")
    output = (
        f"=== Odpowiedź modelu ({model_name}) ===\n"
        f"Pytanie: {state['question']}\n"
        f"Odpowiedź:\n{state['llm_response']}"
    )
    return {"formatted_output": output}

## Pipeline

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

graph.add_node("build_prompt", build_prompt_node)
graph.add_node("call_llm", call_llm_node)
graph.add_node("format_output", format_output_node)

graph.add_edge(START, "build_prompt")
graph.add_edge("build_prompt", "call_llm")
graph.add_edge("call_llm", "format_output")
graph.add_edge("format_output", END)

pipeline = graph.compile()

In [ ]:
initial_state = {"question": "Czym jest LangGraph i do czego się go używa?"}

runtime_config = {
    "configurable": {
        "model_name": "openai/gpt-4o-mini",
        "temperature": 0.5,
        "system_prompt": "Jesteś ekspertem od frameworków AI. Odpowiadaj zwięźle po polsku.",
    }
}

---
## Metoda 1: `invoke` — synchroniczny, wynik na końcu

Blokuje wątek przez cały czas wykonania. Dostajesz kompletny State dopiero gdy wszystkie node'y skończą.

In [ ]:
print("invoke — start")
start = time.time()

result = pipeline.invoke(initial_state, config=runtime_config)

print(f"invoke — wynik po {time.time() - start:.1f}s (wszystkie node'y naraz)\n")
for key, value in result.items():
    print(key, "\n", value, "\n====\n")

In [ ]:
result

---
## Metoda 2: `ainvoke` — asynchroniczny, wynik na końcu

Zwalnia wątek podczas czekania na I/O. Wynik identyczny jak `invoke` — różnica jest niewidoczna w notebooku, ma znaczenie dopiero gdy wiele requestów przychodzi jednocześnie do serwera.

In [ ]:
print("ainvoke — start")
start = time.time()

result = await pipeline.ainvoke(initial_state, config=runtime_config)

print(f"ainvoke — wynik po {time.time() - start:.1f}s (wszystko naraz, ale wątek był wolny)\n")
for key, value in result.items():
    print(key, "\n", value, "\n====\n")

In [ ]:
result

---
## Metoda 3: `stream` — synchroniczny, wynik po każdym nodzie

Blokuje wątek, ale każdy chunk pojawia się natychmiast po zakończeniu danego node'a — nie czekasz na cały pipeline.

In [ ]:
print("stream — start\n")
start = time.time()

for chunk in pipeline.stream(initial_state, config=runtime_config):
    node_name = list(chunk.keys())[0]
    elapsed = time.time() - start
    print(f"[{elapsed:.1f}s] Node '{node_name}' zakończony — output:")
    for key, value in chunk[node_name].items():
        print(f"  {key}: {value}")
    print()

In [ ]:
chunk  # {<nazwa_node'a>: <zwrotka_node'a>}

---
## Metoda 4: `astream` — asynchroniczny, wynik po każdym nodzie

Łączy zalety obu osi: zwalnia wątek podczas I/O i zwraca wyniki etapami.

In [ ]:
print("astream — start\n")
start = time.time()

async for chunk in pipeline.astream(initial_state, config=runtime_config):
    node_name = list(chunk.keys())[0]
    elapsed = time.time() - start
    print(f"[{elapsed:.1f}s] Node '{node_name}' zakończony — output:")
    for key, value in chunk[node_name].items():
        print(f"  {key}: {value}")
    print()

In [ ]:
chunk

---
## Podsumowanie

```
                 zwraca wynik na końcu  | zwraca wynik po każdym nodzie
                 ─────────────────────┬────────────────────────── 
sync                   invoke           │         stream              
async                  ainvoke          │         astream             
                 ─────────────────────┴──────────────────────────
```

- Całkowity **czas wykonania jest taki sam** dla wszystkich czterech
- **sync vs async** — różnica widoczna tylko przy wielu równoległych requestach (np. w FastAPI)
- **invoke vs stream** — różnica widoczna od razu: stream pokazuje postęp etapami
- Pipeline budujemy tak samo niezależnie od tego w jaki sposób go uruchamiamy

> **ZADANIA**